In [ ]:
%pip install -q transformers torch torchvision pillow numpy pandas tqdm scipy

In [ ]:
import os, json, math, random, hashlib, re
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED=42
MODEL_NAME="Salesforce/blip-image-captioning-base"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(MODEL_NAME, device)

In [ ]:
# ===== EDIT THIS PATH =====
BASE_DIR=Path("Enter the path")
IMAGE_DIR=BASE_DIR/"images"/"val"
SALIENCY_DIR=BASE_DIR/"maps"/"val"
OUTPUT_DIR=BASE_DIR/"major_blip_outputs"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
MAX_IMAGES=None
ATTENTION_LAYER=-1
HEAD_REDUCTION="mean"
SAVE_TOKEN_MAPS=True
if not IMAGE_DIR.exists(): raise FileNotFoundError(IMAGE_DIR)
print(IMAGE_DIR, SALIENCY_DIR, OUTPUT_DIR)

In [ ]:
processor=BlipProcessor.from_pretrained(MODEL_NAME)
model=BlipForConditionalGeneration.from_pretrained(MODEL_NAME, attn_implementation="eager")
model.to(device); model.eval()
tokenizer=processor.tokenizer
print("Loaded", MODEL_NAME)

In [ ]:
EPS=1e-12
def normalize_distribution(x,eps=EPS):
    x=np.asarray(x,dtype=np.float64)
    x=np.nan_to_num(x,nan=0.0,posinf=0.0,neginf=0.0)
    x=np.maximum(x,0.0)
    total=x.sum()
    if not np.isfinite(total) or total<=eps:
        x=np.ones_like(x); total=x.sum()
    return x/total

def kl_divergence(p,q,eps=EPS):
    p=normalize_distribution(p,eps); q=normalize_distribution(q,eps)
    return float(np.sum(p*np.log((p+eps)/(q+eps))))

def js_divergence(p,q,eps=EPS):
    p=normalize_distribution(p,eps); q=normalize_distribution(q,eps)
    m=.5*(p+q)
    return .5*kl_divergence(p,m,eps)+.5*kl_divergence(q,m,eps)

def normalized_entropy(p,eps=EPS):
    p=normalize_distribution(p,eps)
    h=-np.sum(p*np.log(p+eps)); mh=np.log(len(p))
    return float(h/mh) if mh>0 else 0.0

In [ ]:
SPECIAL_IDS=set(tokenizer.all_special_ids)
def token_string(token_id):
    return tokenizer.decode([int(token_id)],skip_special_tokens=False,clean_up_tokenization_spaces=False)
def token_records(ids):
    ids=ids[0].detach().cpu().tolist()
    return [{"sequence_position":i,"token_id":int(t),"token":token_string(t),"is_special":int(t) in SPECIAL_IDS} for i,t in enumerate(ids)]

In [ ]:
def vision_grid(vision_outputs):
    n=vision_outputs.last_hidden_state.shape[1]
    for cls_count in (0,1):
        patches=n-cls_count; g=int(math.sqrt(patches))
        if g*g==patches: return cls_count,g,patches
    raise ValueError(f"Cannot infer square grid from {n} vision tokens")

In [ ]:
def process_image_blip(image_path,saliency_path,save_maps=SAVE_TOKEN_MAPS):
    image=Image.open(image_path).convert("RGB")
    pixel_values=processor(images=image,return_tensors="pt")["pixel_values"].to(device)

    with torch.no_grad():
        generated=model.generate(pixel_values=pixel_values,return_dict_in_generate=True)
    generated_ids=generated.sequences
    caption=processor.decode(generated_ids[0],skip_special_tokens=True,clean_up_tokenization_spaces=True)

    with torch.no_grad():
        vision_outputs=model.vision_model(pixel_values=pixel_values,return_dict=True)
    cls_count,grid,patch_count=vision_grid(vision_outputs)

    with torch.no_grad():
        out=model(pixel_values=pixel_values,labels=generated_ids,output_attentions=True,return_dict=True)
    cross=out.cross_attentions
    if cross is None: raise RuntimeError("Cross-attentions were not returned; use eager attention.")
    att=cross[ATTENTION_LAYER]
    if att.ndim!=4: raise ValueError(f"Unexpected attention shape {tuple(att.shape)}")
    att=att[0].mean(dim=0)

    if att.shape[-1]!=vision_outputs.last_hidden_state.shape[1]:
        raise ValueError(f"Attention/vision mismatch: {att.shape[-1]} vs {vision_outputs.last_hidden_state.shape[1]}")
    if cls_count: att=att[:,cls_count:]
    if att.shape[-1]!=patch_count: raise ValueError("Patch-count mismatch after CLS removal")

    sal=Image.open(saliency_path).convert("L").resize((grid,grid),Image.Resampling.BILINEAR)
    sal=normalize_distribution(np.asarray(sal,dtype=np.float64))

    toks=token_records(generated_ids)
    records=[]; maps=[]
    for q in range(att.shape[0]):
        target=q+1
        if target>=len(toks): continue
        raw=att[q].view(grid,grid).detach().cpu().numpy()
        amap=normalize_distribution(raw); t=toks[target]
        records.append({
            "query_position":q,"sequence_position":target,
            "token_id":t["token_id"],"token":t["token"],"is_special":t["is_special"],
            "JS":js_divergence(amap,sal),"KL":kl_divergence(amap,sal),
            "attention_entropy":normalized_entropy(amap)
        })
        maps.append(amap.tolist())

    sem=[r for r in records if not r["is_special"]]
    def mean(k): return float(np.mean([r[k] for r in sem])) if sem else np.nan
    def mx(k): return float(np.max([r[k] for r in sem])) if sem else np.nan

    result={
        "image_id":Path(image_path).stem,"caption":caption,
        "num_generated_tokens":len(toks),"num_attention_tokens":len(records),
        "num_semantic_tokens":len(sem),
        "vision_tokens":int(vision_outputs.last_hidden_state.shape[1]),
        "cls_tokens_removed":int(cls_count),"patch_count":int(patch_count),
        "grid_size":int(grid),"avg_JS":mean("JS"),"avg_KL":mean("KL"),
        "avg_attention_entropy":mean("attention_entropy"),
        "max_JS":mx("JS"),"max_KL":mx("KL"),"token_records":records
    }
    if save_maps: result["_token_maps"]=maps
    return result

In [ ]:
def first_pair():
    files=sorted(list(IMAGE_DIR.glob("*.jpg"))+list(IMAGE_DIR.glob("*.jpeg")))
    for p in files:
        s=SALIENCY_DIR/f"{p.stem}.png"
        if s.exists(): return p,s
    return None,None

smoke_image,smoke_saliency=first_pair()
if smoke_image is None:
    print("No image/saliency pair found")
else:
    r=process_image_blip(smoke_image,smoke_saliency,False)
    print("Image:",r["image_id"])
    print("Caption:",r["caption"])
    print("Grid:",r["grid_size"])
    print("Tokens:",r["num_semantic_tokens"])
    print("JS:",r["avg_JS"],"KL:",r["avg_KL"])

In [ ]:
files=sorted(list(IMAGE_DIR.glob("*.jpg"))+list(IMAGE_DIR.glob("*.jpeg")))
if MAX_IMAGES is not None: files=files[:MAX_IMAGES]
results=[]; failures=[]; missing=[]

for p in tqdm(files,desc="BLIP attention analysis"):
    s=SALIENCY_DIR/f"{p.stem}.png"
    if not s.exists():
        missing.append({"image_id":p.stem,"image_path":str(p),"saliency_path":str(s),"reason":"missing_saliency"})
        continue
    try:
        results.append(process_image_blip(p,s,SAVE_TOKEN_MAPS))
    except Exception as e:
        failures.append({"image_id":p.stem,"image_path":str(p),"saliency_path":str(s),
                         "error_type":type(e).__name__,"error":str(e)})

print("Successful:",len(results),"Missing:",len(missing),"Failures:",len(failures))

In [ ]:
rows=[{k:v for k,v in r.items() if k not in {"token_records","_token_maps"}} for r in results]
df=pd.DataFrame(rows)
if len(df): df=df.sort_values("image_id").reset_index(drop=True)
print(df.head())
print("Rows:",len(df))

In [ ]:
image_out=OUTPUT_DIR/"blip_attention_alignment_results.csv"
token_out=OUTPUT_DIR/"blip_token_attention_results.jsonl"
fail_out=OUTPUT_DIR/"failures.csv"
miss_out=OUTPUT_DIR/"missing_saliency.csv"

df.to_csv(image_out,index=False)

with open(token_out,"w",encoding="utf-8") as f:
    for r in results:
        for i,rec in enumerate(r["token_records"]):
            x={"image_id":r["image_id"],"caption":r["caption"],**rec}
            if SAVE_TOKEN_MAPS: x["attention_map"]=r["_token_maps"][i]
            f.write(json.dumps(x)+"
")

pd.DataFrame(failures).to_csv(fail_out,index=False)
pd.DataFrame(missing).to_csv(miss_out,index=False)

print(image_out)
print(token_out)
print(fail_out)
print(miss_out)

In [ ]:
if len(df):
    summary={
        "n_images":int(len(df)),
        "mean_JS":float(df.avg_JS.mean()),"std_JS":float(df.avg_JS.std()),
        "median_JS":float(df.avg_JS.median()),
        "mean_KL":float(df.avg_KL.mean()),"std_KL":float(df.avg_KL.std()),
        "median_KL":float(df.avg_KL.median()),
        "mean_attention_entropy":float(df.avg_attention_entropy.mean()),
        "mean_semantic_tokens":float(df.num_semantic_tokens.mean())
    }
    print(json.dumps(summary,indent=2))
    with open(OUTPUT_DIR/"dataset_summary.json","w") as f: json.dump(summary,f,indent=2)

In [ ]:
LABEL_FILE=BASE_DIR/"manual_labels.csv"

if LABEL_FILE.exists() and len(df):
    labels=pd.read_csv(LABEL_FILE)
    labels["image_id"]=(labels["image_id"].astype(str)
                        .str.replace(".jpg","",regex=False)
                        .str.replace(".jpeg","",regex=False).str.strip())
    dfl=df.copy()
    dfl["image_id"]=dfl["image_id"].astype(str).str.strip()
    cols=[c for c in ["image_id","generated_caption","error_type","primary_error_token","notes"] if c in labels.columns]
    dfl=dfl.merge(labels[cols],on="image_id",how="inner")
    if "error_type" in dfl.columns: dfl["has_error"]=dfl.error_type!=0
    dfl.to_csv(OUTPUT_DIR/"blip_attention_with_audit_labels.csv",index=False)
    print("Labeled rows:",len(dfl))
else:
    dfl=None
    print("manual_labels.csv not found; skipping label integration")

In [ ]:
def inspect_tokens(image_id):
    out=[]
    with open(token_out,encoding="utf-8") as f:
        for line in f:
            x=json.loads(line)
            if x["image_id"]==image_id: out.append(x)
    return out

if dfl is not None and len(dfl):
    ex=dfl.iloc[0]["image_id"]
    for x in inspect_tokens(ex):
        print(x["sequence_position"],repr(x["token"]),
              "special=",x["is_special"],"JS=",round(x["JS"],5),"KL=",round(x["KL"],5))

In [ ]:
manifest={
    "model_name":MODEL_NAME,"seed":SEED,"device":str(device),
    "attention_layer":ATTENTION_LAYER,"head_reduction":HEAD_REDUCTION,
    "max_images":MAX_IMAGES,"save_token_maps":SAVE_TOKEN_MAPS,
    "image_directory":str(IMAGE_DIR),"saliency_directory":str(SALIENCY_DIR),
    "num_input_images":len(files),"num_successful":len(results),
    "num_missing_saliency":len(missing),"num_failures":len(failures),
    "transformers_version":__import__("transformers").__version__,
    "torch_version":torch.__version__
}
with open(OUTPUT_DIR/"run_manifest.json","w") as f: json.dump(manifest,f,indent=2)
print(json.dumps(manifest,indent=2))